In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import soundfile as sf
import torch
import torch.nn as nn



In [14]:
df_meta = pd.read_csv("data/train.csv")

print(df_meta.head())

  primary_label secondary_labels type  latitude  longitude scientific_name  \
0       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
1       1161364               []   []  -22.7558   -46.8700    Guyalna cuta   
2       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
3       1161364               []   []  -22.7547   -46.8728    Guyalna cuta   
4       1161364               []   []  -22.7426   -46.8985    Guyalna cuta   

    common_name class_name  inat_taxon_id         author   license  rating  \
0  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
1  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
2  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
3  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   
4  Guyalna cuta    Insecta        1161364  Lucas Barbosa  cc-by-nc     0.0   

                                                 url          

In [ ]:
import librosa
class DataProcessor:

    def __init__(self, class_name, input_path, df_meta_subset):
        self.class_name = class_name
        self.input_path = input_path
        self.df_meta_subset = df_meta_subset
        self.labels = np.empty(0, dtype=object)
        self.n_mels = 128
        self.n_fft = 2048
        self.hop_length = 512
    

    def _open_resample_melspec(self, file):
        data, samplerate = sf.read(f'{self.input_path}{file}')
        resampled_data = sp.signal.resample(data, samples)
        mel_spec = librosa.feature.melspectrogram(y=resampled_data, sr=samplerate, 
                                                  n_mels=self.n_mels, n_fft=self.n_fft, hop_length=self.hop_length)
        self.labels = np.append(self.labels, self.df_meta_subset[self.df_meta_subset["filename"] == file]['primary_label'].values[0])
        return mel_spec
    
    def process_files(self):
        self.mel_spec_array = np.array([self._open_resample_melspec(file) for file in self.df_meta_subset["filename"]])

    def save_arrays(self):
        np.save(f"{self.class_name}_mel_spec.npy", self.mel_spec_array)
        np.save(f"{self.class_name}_labels.npy", self.labels)

In [16]:
input_path = 'data/train_audio/'
class_type = 'greyel'
df_meta_subset = df_meta[df_meta['filename'].str.contains(class_type)]

greyel_processor = DataProcessor(class_type, input_path, df_meta_subset)
greyel_processor.process_files()
greyel_processor.save_arrays()

size_mb = greyel_processor.mel_spec_array.nbytes / (1024 ** 2)
print(f"{size_mb:.2f} MB")
print(f"Shape of mel spectrogram array: {greyel_processor.mel_spec_array.shape}")
print(f"Shape of labels array: {greyel_processor.labels.shape}")

453.20 MB
Shape of mel spectrogram array: (475, 128, 977)
Shape of labels array: (475,)


In [17]:

class_type = 'grekis'
df_meta_subset = df_meta[df_meta['filename'].str.contains(class_type)]

grekis_processor = DataProcessor(class_type, input_path, df_meta_subset)
grekis_processor.process_files()
grekis_processor.save_arrays()

size_mb = grekis_processor.mel_spec_array.nbytes / (1024 ** 2)
print(f"{size_mb:.2f} MB")
print(f"Shape of mel spectrogram array: {grekis_processor.mel_spec_array.shape}")
print(f"Shape of labels array: {grekis_processor.labels.shape}")

459.88 MB
Shape of mel spectrogram array: (482, 128, 977)
Shape of labels array: (482,)


In [2]:
data_1 = np.load("greyel_mel_spec.npy")
y_1 = np.load("greyel_labels.npy", allow_pickle=True)

data_2 = np.load("grekis_mel_spec.npy")
y_2 = np.load("grekis_labels.npy", allow_pickle=True)

data = np.vstack((data_1, data_2))
y = np.append(y_1, y_2)

In [3]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [4]:
from sklearn.model_selection import train_test_split
    
X_train, X_test, y_train, y_test = train_test_split(data, y_encoded, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler, normalize
# scaler = StandardScaler()
# scaled_fft_array = scaler.fit_transform(X_train)
# print(X_train[0:5, 0:5])
# X_train = scaled_fft_array
# print(X_train[0:5, 0:5])

normalized_fft_array = normalize(X_train, norm='l2')
print(X_train[0:5, 0:5])
X_train = normalized_fft_array
print(X_train[0:5, 0:5])


In [5]:
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).long()


print(X_train.shape)
X_train = X_train.unsqueeze(1) # Add a channel dimension for CNN input
print(X_train.shape)

X_train.to(device)
y_train.to(device)
train_ds = TensorDataset(X_train, y_train)

batch_size = 20
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)


torch.Size([765, 128, 977])
torch.Size([765, 1, 128, 977])


In [ ]:
model = nn.Sequential()
model.add_module('conv1', nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5, stride=1, padding=2))
model.add_module('relu1', nn.ReLU())
model.add_module('pool1', nn.MaxPool2d(kernel_size=2, stride=2))
model.add_module('conv2', nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2))
model.add_module('relu2', nn.ReLU())
model.add_module('pool2', nn.MaxPool2d(kernel_size=2, stride=2))
model.add_module('flatten', nn.Flatten())
model.add_module('fc1', nn.Linear(499712, 640))
model.add_module('relu3', nn.ReLU())
model.add_module('dropout', nn.Dropout(p=0.5))
model.add_module('fc2', nn.Linear(640, len(label_encoder.classes_)))
model.to(device)
# conv = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=5, stride=1, padding=1)
# for x_batch, y_batch in train_dl:
#     test = model(x_batch.to(device))
#     print(test.shape)


torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([20, 499712])
torch.Size([5, 499712])


In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

torch.manual_seed(1)
num_epochs = 20
for epoch in range(num_epochs):
    accuracy_hist_train = 0
    for x_batch, y_batch in train_dl:
        pred = model(x_batch.to(device))
        pred = pred.squeeze()  # Remove extra dimensions if necessary
        loss = loss_fn(pred, y_batch.to(device))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        is_correct = (torch.argmax(pred, dim=1) == y_batch.to(device)).float()
        accuracy_hist_train += is_correct.sum().cpu()
    accuracy_hist_train /= len(train_dl.dataset)
    print(f'Epoch {epoch}  Accuracy {accuracy_hist_train:.4f}')

RuntimeError: mat1 and mat2 shapes cannot be multiplied (20x7808 and 125056x640)